In [ ]:
import re

import pandas as pd
import numpy as np

df = pd.read_csv("data/ncsl_ai_data_clean.csv")

In [ ]:
document_types = ['Resolution', 'Study Bill',
                  'Memorial', 'Docket', 'Draft', 'Reference']

# Create a list of conditions checking if each type is a substring of chamber_name
conditions = [
    df['chamber_name'].str.lower().str.contains(doc.lower(), na=False)
    for doc in document_types
]

# Apply the conditions in order, defaulting to 'Bill' if none match
df['document_type'] = np.select(conditions, document_types, default='Bill')

In [17]:
# Types of Legislation
df["is_resolution"] = df.chamber_name.str.contains("Resolution")
df["is_joint_resolution"] = df.chamber_name.str.contains("Joint Resolution")
df["is_concurrent_resolution"] = df.chamber_name.str.contains(
    "Concurrent Resolution")
df["is_simple_resolution"] = df.chamber_prefix.isin(["HR", "SR", "AR"])

# Remove full title chamber name row - same as prefix analysis-wise
df = df.drop(columns=["chamber_name"])

In [18]:
df['jurisdiction_type'] = df.state.map(
    lambda s: "Territory" if s in {"PR", "VI", "GU", "AS", "MP"}
    else "District" if s == "DC" else "State")

In [19]:
df["companion_bill_count"] = df.groupby(
    ["state", "session_year", "title", "summary"], dropna=False).bill_id.transform("size")

In [ ]:
df.category = df.category.fillna("").str.lower()
df.category = df.category.str.replace(' ', '_')

# One hot encoding categories from multi-label text
# Split strings by separator, strip whitespace, and create dummy variables
dummies = df.category.fillna("").str.get_dummies(sep=";")
dummies = dummies.rename(columns=lambda c: "category_" +
                         re.sub(r"\W+", "_", c.lower()))

df["num_categories"] = dummies.sum(axis=1)
df = pd.concat([df, dummies], axis=1)
df = df.drop(columns=["category", "category_"])

df.columns

Index(['bill_id', 'state', 'session_year', 'page_year', 'num_pages_listed',
       'chamber_prefix', 'bill_num', 'bill_label', 'is_special_session',
       'title', 'summary', 'status', 'status_type', 'status_stale',
       'outcome_final', 'document_type', 'is_resolution',
       'is_joint_resolution', 'is_concurrent_resolution',
       'is_simple_resolution', 'jurisdiction_type', 'companion_bill_count',
       'num_categories', 'category_appropriations', 'category_audit',
       'category_child_pornography', 'category_criminal_use',
       'category_cybersecurity', 'category_education_training',
       'category_education_use', 'category_effect_on_labor_employment',
       'category_elections', 'category_energy', 'category_government_use',
       'category_health_use', 'category_housing', 'category_impact_assessment',
       'category_judicial_use', 'category_notification',
       'category_oversight_governance', 'category_personhood',
       'category_private_right_of_action', 'cate

In [21]:
df.to_csv("data/ncsl_ai_data_final.csv", index=False)